In [ ]:
# The framework prints a couple of "experimental" notices on import; quiet them.
import warnings
warnings.filterwarnings("ignore", message=r".*experimental.*")

In [ ]:
import subprocess
from pathlib import Path

from dotenv import load_dotenv
from agent_framework import Agent, MCPStdioTool
from agent_framework.openai import OpenAIChatClient

load_dotenv(override=True)

In [ ]:
MODEL = "gpt-5.4-mini"

client = OpenAIChatClient(model=MODEL)

agent = Agent(
    client=client,
    instructions="You are a concise, friendly assistant. Reply in a single short sentence.",
)

In [ ]:
result = await agent.run("Say hello in Spanish.")
print(result.text)

In [ ]:
import board

board.reset_board()
board.add_goal("Read notes.txt, translate its contents into natural Spanish, and write the Spanish to spanish.txt.")
board.list_todos()

In [ ]:
board.show_board()

In [ ]:
def show_todos() -> list[dict]:
    """List every todo on the board. A goal has parent_id None; a step has parent_id set to its goal's id."""
    return board.list_todos()

def plan_steps(goal_id: int, steps: list[str]) -> dict:
    """Break a goal into an ordered checklist of steps on the board. Pass the goal's id and a short list of step descriptions."""
    return {"goal_id": goal_id, "step_ids": [board.add_step(goal_id, step) for step in steps]}

def complete_task(task_id: int, result: str) -> dict:
    """Mark a todo (a step or the goal) with this id as done and record a short result summary."""
    board.complete_todo(task_id, result)
    return {"task_id": task_id, "status": "done"}

In [ ]:
board_agent = Agent(
    client=client,
    instructions="You help manage a shared todo board.",
    tools=[show_todos, complete_task],
)

In [ ]:
result = await board_agent.run("What is on the board right now, and what is its status?")
print(result.text)

In [ ]:
workspace = Path("workspace").resolve()   # the only folder the agent may touch


class FilesystemMCP(MCPStdioTool):
    """The filesystem server with its stderr sent to DEVNULL and its working
    directory set to the workspace, so file names resolve there and it runs
    cleanly from a Jupyter kernel on Windows."""

    def get_mcp_client(self):
        from mcp.client.stdio import StdioServerParameters, stdio_client

        params = StdioServerParameters(command=self.command, args=self.args, env=self.env, cwd=str(workspace))
        return stdio_client(server=params, errlog=subprocess.DEVNULL)


filesystem = FilesystemMCP(
    name="filesystem",
    command="npx",
    args=["-y", "@modelcontextprotocol/server-filesystem", str(workspace)],
)

In [ ]:
file_agent = Agent(
    client=client,
    instructions="You can read and write files in your workspace. Use your tools to do what is asked.",
    tools=[filesystem],
)

In [ ]:
async with filesystem:
    result = await file_agent.run("Read notes.txt and summarize it in one short sentence.")
print(result.text)

In [ ]:
INSTRUCTIONS = """
You are a careful worker with a shared todo board and a set of file tools.

Take the pending goal and see it through. Begin by laying out a short plan: the handful of concrete steps the work itself breaks down into, added to the board under the goal. Then carry them out with your file tools, marking each step done as you finish it. Once the steps are all done, close the goal. Your files live in the single folder your tools are allowed to use.
"""

worker = Agent(
    client=client,
    instructions=INSTRUCTIONS,
    tools=[show_todos, plan_steps, complete_task, filesystem],
)

board.reset_board()
goal_id = board.add_goal("Read notes.txt, translate its contents into natural Spanish, and write the Spanish to spanish.txt.")
board.claim_todo(goal_id)

async with filesystem:
    await worker.run("Please work the pending goal on the board.")
board.show_board()

In [ ]:
import functools
import subprocess
from pathlib import Path

from dotenv import load_dotenv
from agno.agent import Agent
from agno.models.openai import OpenAIChat
from agno.tools.mcp import MCPTools
from mcp import StdioServerParameters

load_dotenv(override=True)

In [ ]:
MODEL = "gpt-5.4-mini"

model = OpenAIChat(id=MODEL)

agent = Agent(
    model=model,
    instructions="You are a concise, friendly assistant. Reply in a single short sentence.",
)

In [ ]:
result = await agent.arun(input="Say hello in Spanish.")
print(result.content)

In [ ]:
import board

board.reset_board()
board.add_goal("Read notes.txt, translate its contents into natural Spanish, and write the Spanish to spanish.txt.")
board.list_todos()

In [ ]:
board.show_board()

In [ ]:
def show_todos() -> list[dict]:
    """List every todo on the board. A goal has parent_id None; a step has parent_id set to its goal's id."""
    return board.list_todos()

def plan_steps(goal_id: int, steps: list[str]) -> dict:
    """Break a goal into an ordered checklist of steps on the board. Pass the goal's id and a short list of step descriptions."""
    return {"goal_id": goal_id, "step_ids": [board.add_step(goal_id, step) for step in steps]}

def complete_task(task_id: int, result: str) -> dict:
    """Mark a todo (a step or the goal) with this id as done and record a short result summary."""
    board.complete_todo(task_id, result)
    return {"task_id": task_id, "status": "done"}

In [ ]:
board_agent = Agent(
    model=model,
    instructions="You help manage a shared todo board.",
    tools=[show_todos, complete_task],
)

In [ ]:
result = await board_agent.arun(input="What is on the board right now, and what is its status?")
print(result.content)

In [ ]:
# # Agno does not expose the MCP server's stderr, so point its stdio client at DEVNULL.
# import agno.tools.mcp.mcp as agno_mcp
# from mcp.client.stdio import stdio_client
# agno_mcp.stdio_client = functools.partial(agno_mcp.stdio_client, errlog=subprocess.DEVNULL)

# workspace = Path("workspace").resolve()   # the only folder the agent may touch

# server = StdioServerParameters(
#     command="npx",
#     args=["-y", "@modelcontextprotocol/server-filesystem", str(workspace)],
#     cwd=str(workspace),
# )


In [ ]:
import subprocess
from pathlib import Path
from functools import partial

from mcp import StdioServerParameters
from mcp.client.stdio import stdio_client

# Suppress stderr output from the MCP server (npx)
stdio_client = partial(stdio_client, errlog=subprocess.DEVNULL)

workspace = Path("workspace").resolve()  # the only folder the agent may touch

server = StdioServerParameters(
    command="npx",
    args=[
        "-y",
        "@modelcontextprotocol/server-filesystem",
        str(workspace),
    ],
    cwd=str(workspace),
)

In [ ]:
import agno.tools.mcp.mcp as agno_mcp

print(agno_mcp.__file__)
print("Has stdio_client:", hasattr(agno_mcp, "stdio_client"))
print("Module attributes:", [x for x in dir(agno_mcp) if "stdio" in x])

In [ ]:
async with MCPTools(server_params=server, timeout_seconds=60) as filesystem:
    file_agent = Agent(
        model=model,
        instructions="You can read and write files in your workspace. Use your tools to do what is asked.",
        tools=[filesystem],
    )
    result = await file_agent.arun(input="Read notes.txt and summarize it in one short sentence.")
print(result.content)

In [ ]:
INSTRUCTIONS = """
You are a careful worker with a shared todo board and a set of file tools.

Take the pending goal and see it through. Begin by laying out a short plan: the handful of concrete steps the work itself breaks down into, added to the board under the goal. Then carry them out with your file tools, marking each step done as you finish it. Once the steps are all done, close the goal. Your files live in the single folder your tools are allowed to use.
"""

board.reset_board()
goal_id = board.add_goal("Read notes.txt, translate its contents into natural Spanish, and write the Spanish to spanish.txt.")
board.claim_todo(goal_id)

async with MCPTools(server_params=server, timeout_seconds=60) as filesystem:
    worker = Agent(
        model=model,
        instructions=INSTRUCTIONS,
        tools=[show_todos, plan_steps, complete_task, filesystem],
    )
    await worker.arun(input="Please work the pending goal on the board.")
board.show_board()

In [ ]:
import time

start = time.perf_counter()
agents = [Agent(model=model, tools=[show_todos]) for _ in range(1000)]
elapsed = time.perf_counter() - start
print(f"Built {len(agents)} agents in {elapsed * 1000:.1f} ms")
print(f"That is about {elapsed / len(agents) * 1_000_000:.1f} microseconds each.")